
# TP CNN — LeNet-5 adapté à la base AMHCD RGB

**Objectif :** reprogrammer et adapter **LeNet-5** à la base **AMHCD (Amazigh Handwritten Character Database)** en RGB pour classifier **33 caractères Tifinagh**.

Ce notebook couvre :
- inspection et nettoyage logique du dataset ;
- suppression du problème de duplication présent dans l'archive fournie ;
- séparation **Train / Validation / Test** stratifiée ;
- adaptation de LeNet-5 aux images **RGB** ;
- fonctions paramétrables pour convolution, pooling et couches fully connected ;
- entraînement avec **SGD** et **Adam** ;
- courbes **loss / accuracy** ;
- métriques quantitatives ;
- matrice de confusion ;
- résultats qualitatifs ;
- visualisation des **feature maps** ;
- export des résultats pour l'article scientifique.

> **Important :** l'archive fournie contient deux copies identiques du dataset sous des chemins qui diffèrent seulement par la casse (`AMHCD_64/...` et `amhcd_64/...`).  
> Le notebook utilise uniquement une copie afin d'éviter la duplication et le *data leakage*.


In [ ]:

# 1) Imports et configuration générale

import os
import json
import time
import random
import zipfile
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_recall_fscore_support
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch :", torch.__version__)
print("Device  :", DEVICE)
if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))



## 2. Paramètres de l'expérience

L'architecture LeNet-5 historique travaille avec des images de taille **32×32**.  
La base AMHCD fournie contient des images **64×64 RGB**.

Nous conservons l'information couleur (`3 channels`) mais nous redimensionnons les images à **32×32** afin de rester proches de l'architecture LeNet-5.


In [ ]:

# 2) Paramètres

ZIP_PATH = Path("archive(1).zip")     # Modifier si nécessaire
WORK_DIR = Path("amhcd_work")
DATA_DIR = WORK_DIR / "AMHCD_64"

IMAGE_SIZE = 32
BATCH_SIZE = 128
NUM_WORKERS = 0   # Valeur sûre sous Windows/Jupyter. Mettre 2 ou 4 si souhaité.
EPOCHS = 15
LEARNING_RATE = 1e-3

RESULTS_DIR = Path("results_lenet5_amhcd")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("ZIP_PATH    :", ZIP_PATH.resolve())
print("WORK_DIR    :", WORK_DIR.resolve())
print("RESULTS_DIR :", RESULTS_DIR.resolve())



## 3. Extraction d'une seule copie du dataset

L'archive contient deux arbres identiques :
- `AMHCD_64/AMHCD_64/...`
- `amhcd_64/AMHCD_64/...`

Pour éviter de doubler artificiellement les données, nous extrayons uniquement le premier arbre.


In [ ]:

# 3) Extraction sélective : une seule copie des images

if not ZIP_PATH.exists():
    raise FileNotFoundError(
        f"Archive introuvable : {ZIP_PATH}\n"
        "Placez archive(1).zip dans le même dossier que le notebook "
        "ou modifiez ZIP_PATH."
    )

if not DATA_DIR.exists():
    WORK_DIR.mkdir(parents=True, exist_ok=True)

    prefix = "AMHCD_64/AMHCD_64/"
    extracted = 0

    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        for member in zf.infolist():
            name = member.filename
            if name.startswith(prefix) and name.lower().endswith((".jpg", ".jpeg", ".png")):
                relative = Path(name[len(prefix):])
                target = DATA_DIR / relative
                target.parent.mkdir(parents=True, exist_ok=True)

                with zf.open(member) as src, open(target, "wb") as dst:
                    shutil.copyfileobj(src, dst)

                extracted += 1

    print(f"{extracted} images extraites.")
else:
    print("Dataset déjà extrait :", DATA_DIR)

print("Dataset root :", DATA_DIR.resolve())



## 4. Vérification du dataset

On vérifie :
- le nombre de classes ;
- le nombre d'images ;
- l'équilibre des classes ;
- les dimensions et le mode RGB.


In [ ]:

# 4) Statistiques du dataset

class_dirs = sorted([p for p in DATA_DIR.iterdir() if p.is_dir()])
class_names = [p.name for p in class_dirs]

counts = {}
for cls_dir in class_dirs:
    counts[cls_dir.name] = len([
        p for p in cls_dir.iterdir()
        if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
    ])

counts_df = pd.DataFrame({
    "classe": list(counts.keys()),
    "nombre_images": list(counts.values())
})

print("Nombre de classes :", len(class_names))
print("Nombre total d'images :", counts_df["nombre_images"].sum())
print("Min / classe :", counts_df["nombre_images"].min())
print("Max / classe :", counts_df["nombre_images"].max())

display(counts_df)


In [ ]:

# Vérification d'une image

sample_path = next(DATA_DIR.rglob("*.jpeg"))
sample_img = Image.open(sample_path)

print("Exemple :", sample_path)
print("Taille  :", sample_img.size)
print("Mode    :", sample_img.mode)

plt.figure(figsize=(3, 3))
plt.imshow(sample_img)
plt.title(sample_path.parent.name)
plt.axis("off")
plt.show()



## 5. Correspondance Latin ↔ Tifinagh

La base utilise des noms de dossiers latinisés (`ya`, `yab`, ...).  
On définit la correspondance avec les 33 caractères Tifinagh fournis avec le dataset.


In [ ]:

# 5) Labels AMHCD

LATIN_LABELS = [
    "ya", "yab", "yach", "yad", "yadd", "yae", "yaf", "yag", "yagh", "yagw",
    "yah", "yahh", "yaj", "yak", "yakw", "yal", "yam", "yan", "yaq", "yar",
    "yarr", "yas", "yass", "yat", "yatt", "yaw", "yax", "yay", "yaz", "yazz",
    "yey", "yi", "yu"
]

TIFINAGH_LABELS = [
    "ⴰ", "ⴱ", "ⵛ", "ⴷ", "ⴹ", "ⵄ", "ⴼ", "ⴳ", "ⵖ", "ⴳⵯ",
    "ⵀ", "ⵃ", "ⵊ", "ⴽ", "ⴽⵯ", "ⵍ", "ⵎ", "ⵏ", "ⵇ", "ⵔ",
    "ⵕ", "ⵙ", "ⵚ", "ⵜ", "ⵟ", "ⵡ", "ⵅ", "ⵢ", "ⵣ", "ⵥ",
    "ⴻ", "ⵉ", "ⵓ"
]

latin_to_tifinagh = dict(zip(LATIN_LABELS, TIFINAGH_LABELS))

mapping_df = pd.DataFrame({
    "latin": LATIN_LABELS,
    "tifinagh": TIFINAGH_LABELS
})

display(mapping_df)



## 6. Création des splits Train / Validation / Test

Nous utilisons un split **80% / 10% / 10%**, stratifié par classe :
- 80% entraînement ;
- 10% validation ;
- 10% test.

Le jeu de test reste totalement indépendant jusqu'à l'évaluation finale.


In [ ]:

# 6) Dataset brut pour récupérer chemins et labels

base_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor()
])

full_dataset = datasets.ImageFolder(
    root=DATA_DIR,
    transform=base_transform
)

targets = np.array(full_dataset.targets)
indices = np.arange(len(full_dataset))

train_idx, temp_idx = train_test_split(
    indices,
    test_size=0.20,
    random_state=SEED,
    stratify=targets
)

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    random_state=SEED,
    stratify=targets[temp_idx]
)

print("Total :", len(full_dataset))
print("Train :", len(train_idx))
print("Val   :", len(val_idx))
print("Test  :", len(test_idx))

assert len(set(train_idx) & set(val_idx)) == 0
assert len(set(train_idx) & set(test_idx)) == 0
assert len(set(val_idx) & set(test_idx)) == 0



## 7. Calcul de la moyenne et de l'écart-type sur le Train uniquement

Pour éviter toute fuite d'information, les statistiques de normalisation sont calculées **uniquement sur le jeu d'entraînement**.


In [ ]:

# 7) Mean / Std calculés uniquement sur Train

train_raw = Subset(full_dataset, train_idx)

raw_loader = DataLoader(
    train_raw,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

channel_sum = torch.zeros(3)
channel_sq_sum = torch.zeros(3)
num_pixels = 0

for images, _ in raw_loader:
    b, c, h, w = images.shape
    channel_sum += images.sum(dim=[0, 2, 3])
    channel_sq_sum += (images ** 2).sum(dim=[0, 2, 3])
    num_pixels += b * h * w

mean = channel_sum / num_pixels
std = torch.sqrt(channel_sq_sum / num_pixels - mean ** 2)

MEAN = mean.tolist()
STD = std.tolist()

print("Mean RGB :", MEAN)
print("Std  RGB :", STD)



## 8. DataLoaders normalisés

Le modèle de base reste volontairement simple afin de respecter LeNet-5.  
La data augmentation sera proposée séparément comme amélioration bonus.


In [ ]:

# 8) Transforms définitifs

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

train_dataset_full = datasets.ImageFolder(DATA_DIR, transform=train_transform)
eval_dataset_full = datasets.ImageFolder(DATA_DIR, transform=eval_transform)

train_dataset = Subset(train_dataset_full, train_idx)
val_dataset   = Subset(eval_dataset_full, val_idx)
test_dataset  = Subset(eval_dataset_full, test_idx)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

class_names = train_dataset_full.classes
class_to_idx = train_dataset_full.class_to_idx

print("Classes :", len(class_names))
print(class_to_idx)


In [ ]:

# Visualisation de quelques exemples non normalisés

fig, axes = plt.subplots(2, 5, figsize=(12, 5))

for ax, idx in zip(axes.flat, train_idx[:10]):
    path, label = full_dataset.samples[idx]
    img = Image.open(path).convert("RGB")
    ax.imshow(img)
    latin = full_dataset.classes[label]
    tifinagh = latin_to_tifinagh.get(latin, "?")
    ax.set_title(f"{latin} / {tifinagh}")
    ax.axis("off")

plt.tight_layout()
plt.show()



# 9. Couches paramétrables

Le sujet demande d'implémenter les couches sous forme de fonctions paramétrables.  
On crée donc des fonctions pour :
- convolution ;
- pooling ;
- fully connected.


In [ ]:

# 9) Fonctions paramétrables

def couche_convolution(
    in_channels,
    out_channels,
    kernel_size=5,
    stride=1,
    padding=0
):
    return nn.Conv2d(
        in_channels=in_channels,
        out_channels=out_channels,
        kernel_size=kernel_size,
        stride=stride,
        padding=padding
    )


def couche_pooling(
    type_pool="avg",
    kernel_size=2,
    stride=2
):
    if type_pool.lower() == "avg":
        return nn.AvgPool2d(kernel_size=kernel_size, stride=stride)

    if type_pool.lower() == "max":
        return nn.MaxPool2d(kernel_size=kernel_size, stride=stride)

    raise ValueError("type_pool doit être 'avg' ou 'max'.")


def couche_fully_connected(in_features, out_features):
    return nn.Linear(in_features, out_features)



# 10. Architecture LeNet-5 RGB

Adaptation principale :
- LeNet-5 historique : entrée `32×32×1`
- Notre version : entrée `32×32×3`

Les dimensions deviennent :

`32×32×3 → Conv(6,5×5) → 28×28×6 → AvgPool → 14×14×6 → Conv(16,5×5) → 10×10×16 → AvgPool → 5×5×16 → Flatten(400) → 120 → 84 → 33`


In [ ]:

# 10) LeNet-5 RGB

class LeNet5RGB(nn.Module):
    def __init__(
        self,
        num_classes=33,
        in_channels=3,
        activation="tanh",
        pool_type="avg"
    ):
        super().__init__()

        self.C1 = couche_convolution(
            in_channels=in_channels,
            out_channels=6,
            kernel_size=5
        )

        self.S2 = couche_pooling(
            type_pool=pool_type,
            kernel_size=2,
            stride=2
        )

        self.C3 = couche_convolution(
            in_channels=6,
            out_channels=16,
            kernel_size=5
        )

        self.S4 = couche_pooling(
            type_pool=pool_type,
            kernel_size=2,
            stride=2
        )

        self.C5 = couche_fully_connected(16 * 5 * 5, 120)
        self.F6 = couche_fully_connected(120, 84)
        self.output = couche_fully_connected(84, num_classes)

        if activation.lower() == "tanh":
            self.activation = nn.Tanh()
        elif activation.lower() == "relu":
            self.activation = nn.ReLU()
        else:
            raise ValueError("activation doit être 'tanh' ou 'relu'.")

    def forward(self, x):
        x = self.activation(self.C1(x))
        x = self.S2(x)

        x = self.activation(self.C3(x))
        x = self.S4(x)

        x = torch.flatten(x, 1)

        x = self.activation(self.C5(x))
        x = self.activation(self.F6(x))

        logits = self.output(x)
        return logits


model = LeNet5RGB(
    num_classes=len(class_names),
    in_channels=3,
    activation="tanh",
    pool_type="avg"
).to(DEVICE)

print(model)

dummy = torch.randn(4, 3, 32, 32).to(DEVICE)
with torch.no_grad():
    out = model(dummy)

print("Input :", dummy.shape)
print("Output:", out.shape)
assert out.shape == (4, 33)


In [ ]:

# Nombre de paramètres

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Paramètres totaux      : {total_params:,}")
print(f"Paramètres entraînables: {trainable_params:,}")



# 11. Fonctions d'apprentissage et d'évaluation


In [ ]:

# 11) Boucles train / validation / test

criterion = nn.CrossEntropyLoss()


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(images)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    all_labels = []
    all_preds = []
    all_probs = []

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        loss = criterion(logits, labels)

        probs = torch.softmax(logits, dim=1)
        preds = logits.argmax(dim=1)

        running_loss += loss.item() * images.size(0)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return {
        "loss": epoch_loss,
        "accuracy": epoch_acc,
        "labels": np.array(all_labels),
        "preds": np.array(all_preds),
        "probs": np.array(all_probs)
    }


In [ ]:

def creer_optimiseur(model, nom="adam", lr=1e-3):
    nom = nom.lower()

    if nom == "sgd":
        return torch.optim.SGD(
            model.parameters(),
            lr=lr,
            momentum=0.9
        )

    if nom == "adam":
        return torch.optim.Adam(
            model.parameters(),
            lr=lr
        )

    raise ValueError("Optimiseur supporté : 'sgd' ou 'adam'.")


In [ ]:

def entrainer_modele(
    optimizer_name,
    epochs=EPOCHS,
    lr=LEARNING_RATE,
    activation="tanh",
    pool_type="avg"
):
    model = LeNet5RGB(
        num_classes=len(class_names),
        in_channels=3,
        activation=activation,
        pool_type=pool_type
    ).to(DEVICE)

    optimizer = creer_optimiseur(
        model,
        nom=optimizer_name,
        lr=lr
    )

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }

    best_val_acc = -1.0
    best_state = None

    start = time.time()

    for epoch in range(1, epochs + 1):

        train_loss, train_acc = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            DEVICE
        )

        val_metrics = evaluate(
            model,
            val_loader,
            criterion,
            DEVICE
        )

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_metrics["loss"])
        history["val_acc"].append(val_metrics["accuracy"])

        if val_metrics["accuracy"] > best_val_acc:
            best_val_acc = val_metrics["accuracy"]

            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }

        print(
            f"[{optimizer_name.upper()}] "
            f"Epoch {epoch:02d}/{epochs} | "
            f"Train loss={train_loss:.4f} | "
            f"Train acc={train_acc*100:.2f}% | "
            f"Val loss={val_metrics['loss']:.4f} | "
            f"Val acc={val_metrics['accuracy']*100:.2f}%"
        )

    elapsed = time.time() - start

    model.load_state_dict(best_state)
    model = model.to(DEVICE)

    return model, history, elapsed



# 12. Expérience 1 — SGD

> Si vous faites un test rapide sur CPU, vous pouvez temporairement réduire `EPOCHS`.  
> Pour les résultats de l'article, utilisez le même nombre d'époques pour SGD et Adam.


In [ ]:

sgd_model, sgd_history, sgd_time = entrainer_modele(
    optimizer_name="sgd",
    epochs=EPOCHS,
    lr=0.01
)

torch.save(
    sgd_model.state_dict(),
    RESULTS_DIR / "lenet5_rgb_sgd_best.pt"
)

print(f"Temps SGD : {sgd_time:.2f} secondes")



# 13. Expérience 2 — Adam


In [ ]:

adam_model, adam_history, adam_time = entrainer_modele(
    optimizer_name="adam",
    epochs=EPOCHS,
    lr=1e-3
)

torch.save(
    adam_model.state_dict(),
    RESULTS_DIR / "lenet5_rgb_adam_best.pt"
)

print(f"Temps Adam : {adam_time:.2f} secondes")



# 14. Courbes Loss / Accuracy


In [ ]:

def plot_history(history, title, save_name=None):
    epochs_range = range(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(8, 5))
    plt.plot(epochs_range, history["train_loss"], label="Train Loss")
    plt.plot(epochs_range, history["val_loss"], label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{title} — Loss")
    plt.legend()
    plt.grid(alpha=0.3)

    if save_name:
        plt.savefig(
            RESULTS_DIR / f"{save_name}_loss.png",
            dpi=200,
            bbox_inches="tight"
        )

    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(
        epochs_range,
        np.array(history["train_acc"]) * 100,
        label="Train Accuracy"
    )
    plt.plot(
        epochs_range,
        np.array(history["val_acc"]) * 100,
        label="Validation Accuracy"
    )
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy (%)")
    plt.title(f"{title} — Accuracy")
    plt.legend()
    plt.grid(alpha=0.3)

    if save_name:
        plt.savefig(
            RESULTS_DIR / f"{save_name}_accuracy.png",
            dpi=200,
            bbox_inches="tight"
        )

    plt.show()


plot_history(
    sgd_history,
    "LeNet-5 RGB + SGD",
    "sgd"
)

plot_history(
    adam_history,
    "LeNet-5 RGB + Adam",
    "adam"
)



# 15. Comparaison quantitative SGD vs Adam

Le choix du meilleur modèle est effectué sur la **validation**, jamais sur le test.


In [ ]:

comparison_df = pd.DataFrame([
    {
        "optimizer": "SGD",
        "best_val_accuracy": max(sgd_history["val_acc"]),
        "best_val_loss": min(sgd_history["val_loss"]),
        "training_time_sec": sgd_time
    },
    {
        "optimizer": "Adam",
        "best_val_accuracy": max(adam_history["val_acc"]),
        "best_val_loss": min(adam_history["val_loss"]),
        "training_time_sec": adam_time
    }
])

comparison_df["best_val_accuracy_pct"] = (
    comparison_df["best_val_accuracy"] * 100
)

display(comparison_df)

comparison_df.to_csv(
    RESULTS_DIR / "comparaison_sgd_adam.csv",
    index=False
)


In [ ]:

# Sélection du meilleur modèle selon la meilleure accuracy de validation

if max(adam_history["val_acc"]) >= max(sgd_history["val_acc"]):
    best_model = adam_model
    best_optimizer_name = "Adam"
else:
    best_model = sgd_model
    best_optimizer_name = "SGD"

print("Meilleur modèle selon Validation Accuracy :", best_optimizer_name)



# 16. Évaluation finale sur Test

Le test n'est utilisé qu'après le choix définitif du modèle.


In [ ]:

test_metrics = evaluate(
    best_model,
    test_loader,
    criterion,
    DEVICE
)

y_true = test_metrics["labels"]
y_pred = test_metrics["preds"]
y_prob = test_metrics["probs"]

precision, recall, f1, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    average="macro",
    zero_division=0
)

final_metrics = {
    "best_optimizer": best_optimizer_name,
    "test_loss": float(test_metrics["loss"]),
    "test_accuracy": float(test_metrics["accuracy"]),
    "macro_precision": float(precision),
    "macro_recall": float(recall),
    "macro_f1": float(f1)
}

print(json.dumps(final_metrics, indent=2, ensure_ascii=False))

with open(
    RESULTS_DIR / "final_metrics.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        final_metrics,
        f,
        indent=2,
        ensure_ascii=False
    )



# 17. Classification Report


In [ ]:

report = classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).transpose()
display(report_df)

report_df.to_csv(
    RESULTS_DIR / "classification_report.csv"
)



# 18. Matrice de confusion


In [ ]:

cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(15, 15))

im = ax.imshow(cm)

ax.set_title(
    f"Matrice de confusion — LeNet-5 RGB ({best_optimizer_name})"
)
ax.set_xlabel("Classe prédite")
ax.set_ylabel("Classe réelle")

ax.set_xticks(np.arange(len(class_names)))
ax.set_yticks(np.arange(len(class_names)))

ax.set_xticklabels(class_names, rotation=90)
ax.set_yticklabels(class_names)

fig.colorbar(im, ax=ax)

plt.tight_layout()

plt.savefig(
    RESULTS_DIR / "confusion_matrix.png",
    dpi=220,
    bbox_inches="tight"
)

plt.show()



# 19. Résultats qualitatifs

On affiche :
- des prédictions correctes ;
- des erreurs ;
- la confiance du modèle.


In [ ]:

@torch.no_grad()
def collect_predictions_with_paths(model, subset, n=None):
    model.eval()

    records = []

    indices_to_use = subset.indices
    if n is not None:
        indices_to_use = indices_to_use[:n]

    for global_idx in indices_to_use:
        path, label = eval_dataset_full.samples[global_idx]

        image = Image.open(path).convert("RGB")
        tensor = eval_transform(image).unsqueeze(0).to(DEVICE)

        logits = model(tensor)
        probs = torch.softmax(logits, dim=1)[0]

        pred = int(probs.argmax().item())
        confidence = float(probs[pred].item())

        records.append({
            "path": path,
            "true": label,
            "pred": pred,
            "confidence": confidence
        })

    return records


qualitative_records = collect_predictions_with_paths(
    best_model,
    test_dataset
)

correct_records = [
    r for r in qualitative_records
    if r["true"] == r["pred"]
]

wrong_records = [
    r for r in qualitative_records
    if r["true"] != r["pred"]
]

print("Prédictions correctes :", len(correct_records))
print("Erreurs               :", len(wrong_records))


In [ ]:

def show_records(records, title, n=10):
    records = records[:n]

    cols = 5
    rows = int(np.ceil(len(records) / cols))

    plt.figure(figsize=(14, 3 * rows))

    for i, rec in enumerate(records, 1):
        img = Image.open(rec["path"]).convert("RGB")

        true_latin = class_names[rec["true"]]
        pred_latin = class_names[rec["pred"]]

        true_tif = latin_to_tifinagh.get(true_latin, "?")
        pred_tif = latin_to_tifinagh.get(pred_latin, "?")

        plt.subplot(rows, cols, i)
        plt.imshow(img)

        plt.title(
            f"Vrai: {true_latin} ({true_tif})\n"
            f"Prédit: {pred_latin} ({pred_tif})\n"
            f"Conf: {rec['confidence']*100:.1f}%"
        )

        plt.axis("off")

    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()


show_records(
    correct_records,
    "Exemples correctement classifiés",
    n=10
)

if len(wrong_records) > 0:
    show_records(
        wrong_records,
        "Exemples mal classifiés",
        n=min(10, len(wrong_records))
    )



# 20. Visualisation des Feature Maps

Cette partie permet d'observer ce que détectent les couches convolutionnelles C1 et C3.


In [ ]:

def extraire_feature_maps(model, input_tensor):
    features = {}

    def hook_c1(module, inp, out):
        features["C1"] = out.detach().cpu()

    def hook_c3(module, inp, out):
        features["C3"] = out.detach().cpu()

    h1 = model.C1.register_forward_hook(hook_c1)
    h3 = model.C3.register_forward_hook(hook_c3)

    model.eval()

    with torch.no_grad():
        _ = model(input_tensor.to(DEVICE))

    h1.remove()
    h3.remove()

    return features


In [ ]:

# Une image du test

global_idx = test_dataset.indices[0]
path, label = eval_dataset_full.samples[global_idx]

original = Image.open(path).convert("RGB")
input_tensor = eval_transform(original).unsqueeze(0)

feature_maps = extraire_feature_maps(
    best_model,
    input_tensor
)

plt.figure(figsize=(3, 3))
plt.imshow(original)
plt.title(
    f"Entrée : {class_names[label]} / "
    f"{latin_to_tifinagh.get(class_names[label], '?')}"
)
plt.axis("off")
plt.show()

# C1 : 6 cartes
c1_maps = feature_maps["C1"][0]

plt.figure(figsize=(12, 6))

for i in range(c1_maps.shape[0]):
    plt.subplot(2, 3, i + 1)
    plt.imshow(c1_maps[i].numpy())
    plt.title(f"C1 — filtre {i+1}")
    plt.axis("off")

plt.tight_layout()

plt.savefig(
    RESULTS_DIR / "feature_maps_C1.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

# C3 : 16 cartes
c3_maps = feature_maps["C3"][0]

plt.figure(figsize=(12, 12))

for i in range(c3_maps.shape[0]):
    plt.subplot(4, 4, i + 1)
    plt.imshow(c3_maps[i].numpy())
    plt.title(f"C3 — filtre {i+1}")
    plt.axis("off")

plt.tight_layout()

plt.savefig(
    RESULTS_DIR / "feature_maps_C3.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()



# 21. Bonus — LeNet-5 amélioré

Cette section est **optionnelle** et doit être présentée comme une amélioration séparée du modèle de base.

Améliorations possibles :
- ReLU ;
- MaxPooling ;
- Batch Normalization ;
- Dropout ;
- Data Augmentation.

Le modèle de base doit rester disponible pour permettre une comparaison scientifique correcte.


In [ ]:

# Modèle bonus : exemple d'amélioration

class LeNet5Improved(nn.Module):
    def __init__(self, num_classes=33):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 6, kernel_size=5),
            nn.BatchNorm2d(6),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(6, 16, kernel_size=5),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 5 * 5, 120),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(120, 84),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(84, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


bonus_model = LeNet5Improved(
    num_classes=len(class_names)
).to(DEVICE)

print(bonus_model)



## 22. Résumé automatiquement exploitable dans l'article

Après exécution complète, cette cellule produit une synthèse directement réutilisable dans la section **Résultats** de l'article.


In [ ]:

summary = {
    "dataset": {
        "name": "AMHCD",
        "image_mode": "RGB",
        "original_size": "64x64",
        "model_input_size": "32x32x3",
        "unique_images": len(full_dataset),
        "classes": len(class_names),
        "train_size": len(train_dataset),
        "validation_size": len(val_dataset),
        "test_size": len(test_dataset)
    },
    "architecture": {
        "name": "LeNet-5 RGB",
        "input": "32x32x3",
        "conv1": "6 filtres 5x5",
        "pool1": "Average Pooling 2x2",
        "conv2": "16 filtres 5x5",
        "pool2": "Average Pooling 2x2",
        "fc1": 120,
        "fc2": 84,
        "output": len(class_names),
        "parameters": total_params
    },
    "best_model": final_metrics
}

print(json.dumps(summary, indent=2, ensure_ascii=False))

with open(
    RESULTS_DIR / "article_summary.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        summary,
        f,
        indent=2,
        ensure_ascii=False
    )



# 23. Formules à présenter dans l'article

### Convolution

\[
Y_k(i,j)=b_k+\sum_c\sum_m\sum_n
X_c(i+m,j+n)K_{k,c}(m,n)
\]

### Fonction d'activation Tanh

\[
\tanh(x)=\frac{e^x-e^{-x}}{e^x+e^{-x}}
\]

### Softmax

\[
P(y=c\mid x)=\frac{e^{z_c}}
{\sum_{j=1}^{33}e^{z_j}}
\]

### Entropie croisée

\[
\mathcal{L}
=
-\frac{1}{N}
\sum_{i=1}^{N}
\log P(y_i\mid x_i)
\]

### SGD

\[
\theta_{t+1}
=
\theta_t-\eta\nabla_\theta\mathcal{L}
\]

### Adam

\[
m_t=\beta_1m_{t-1}+(1-\beta_1)g_t
\]

\[
v_t=\beta_2v_{t-1}+(1-\beta_2)g_t^2
\]

\[
\theta_{t+1}
=
\theta_t-\eta
\frac{\hat m_t}
{\sqrt{\hat v_t}+\epsilon}
\]



# Conclusion

Après exécution, le dossier `results_lenet5_amhcd/` doit contenir notamment :

- `lenet5_rgb_sgd_best.pt`
- `lenet5_rgb_adam_best.pt`
- `sgd_loss.png`
- `sgd_accuracy.png`
- `adam_loss.png`
- `adam_accuracy.png`
- `confusion_matrix.png`
- `feature_maps_C1.png`
- `feature_maps_C3.png`
- `classification_report.csv`
- `comparaison_sgd_adam.csv`
- `final_metrics.json`
- `article_summary.json`

Ces éléments fourniront les **résultats quantitatifs et qualitatifs réels** nécessaires pour rédiger l'article IMRAD.
